# 24. Can a forgotten task be brought back?

This is the question the whole project is for. Unlearning drops task 3 to exactly 10%
and stops the model responding to its input. Is the knowledge gone, or only unreachable?

### Why this is not obvious either way

The weights are not stored. A shared **generator** network produces them from a 32-number
**task code**. Unlearning trains only the generator; the task codes are never touched. So
task 3's code still exists exactly as it was, and the generator still serves seven other
tasks well. Nothing was deleted. One mapping was bent out of shape.

Against that, the model now gives the same answer to every image, and that can make the
learning signal vanish: the error has to travel back through near-zero layers that shrink
it on the way.

### The design, and why it needs three arms

Showing that task 3 comes back is not enough on its own. Task 3 might simply be easy to
learn from a handful of images. The question is whether it comes back **faster than it
would from nothing**, which is what would show that something survived.

Every arm starts from the same unlearned model and gets the same small sample of task 3
images and the same number of training steps.

| arm | task code | BatchNorm statistics | what it isolates |
| --- | --- | --- | --- |
| **attacker** | task 3's original | task 3's original | everything an attacker inherits |
| **fresh start** | a brand new code | reset | learning task 3 from nothing, in this same generator |
| **code only** | task 3's original | reset | how much the old code alone carries |

**BatchNorm statistics** are running averages a network keeps about the data it has seen.
They matter here because unlearning never touches them: the repository's own code says so,
and the paper's objective covers generated parameters only. So the statistics still carry
information derived from task 3's images. The third arm measures how much that matters.

### How to read the result

- **attacker clearly above fresh start** → the information survived unlearning. A privacy
  claim for this method does not hold.
- **attacker close to fresh start** → no usable trace is left, and unlearning did remove
  the task-specific mapping.
- **nothing learns in any arm** → the learning signal has vanished. That is ambiguous, not
  evidence of removal, and the write-up must say so.
- **attacker above code only** → a meaningful part of what survived is in the BatchNorm
  statistics, which unlearning does not even attempt to remove. That is worth reporting on
  its own.

### Scope

One seed, one forgotten task, one order. The recovery images come from task 3's own
training set, so this is the question "can someone holding a few of the original images
restore the task", not "does the model generalise again from unseen data".

## 1. Load the fixed code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "65e2e7c850fd6f1852a99d9e81780c27f799c926"
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading this fixed code revision.")
if IN_COLAB:
    REPO = Path("/content/cark-paired-study")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/CARK.git", str(REPO)], check=True)
    if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True).strip():
        raise RuntimeError("Checkout has edits. Use a fresh runtime.")
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_VERSION], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a CARK checkout.")
    actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    implementation = ["uncle", "pyproject.toml", "requirements.txt"]
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--", *implementation], text=True).strip()
    # A later documentation-only commit may contain the same fixed implementation.
    differences = subprocess.check_output(["git", "-C", str(REPO), "diff", CODE_VERSION, actual, "--", *implementation], text=True).strip()
    if dirty or differences:
        raise RuntimeError("Use a clean checkout of CODE_VERSION, or open in Colab. Local training edits are preserved.")
for package in ("boto3", "tqdm", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage and extras dependencies before running this notebook.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))
from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle import checkpoint
from uncle.config import Config
from uncle.hypernet import build_target, HyperNetwork
from uncle.trainer import UnCLe
from uncle.streams import build_tasks
from uncle.study_training import restore_rng, run_request
# The progress panel only draws bars, so it cannot change a result. It
# therefore sits outside the pinned implementation, in scripts/, and this
# checkout is detached to CODE_VERSION, which predates it. Try the checkout
# first, then main, then fall back to ordinary bars rather than failing.
sys.path.insert(0, str(REPO / 'scripts'))
try:
    from dashboard import Dashboard
except ImportError:
    try:
        tools = Path.cwd() / '_tools'
        tools.mkdir(exist_ok=True)
        subprocess.run(['git', '-C', str(REPO), 'fetch', '--quiet', 'origin', 'main'], check=True)
        (tools / 'dashboard.py').write_bytes(subprocess.check_output(
            ['git', '-C', str(REPO), 'show', 'origin/main:scripts/dashboard.py']))
        sys.path.insert(0, str(tools))
        from dashboard import Dashboard
        print('Progress panel fetched from main.')
    except Exception as problem:
        print('Progress panel unavailable, using plain bars:', problem)

        class Dashboard:
            """Just enough of the panel to keep the notebook running."""

            def __init__(self, *args, **kwargs): pass
            def __enter__(self): return self
            def __exit__(self, *args): return False
            def note(self, text): print(text, flush=True)

            def bar(self, total, description, unit='step'):
                from uncle.telemetry import progress_bar
                return progress_bar(total, description)

import torch
print("Fixed implementation revision:", CODE_VERSION)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)

## 2. Settings

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"

ORDER = "order_01"
SEED = 0
FORGET_TASK = "3"

# Unlearn exactly as your study does, then try to recover from there.
UNLEARN_STEPS = 100

# How many images per class the recovery gets. Ten classes, so 20 means 200
# images in total. These come from task 3's own training set.
BUDGETS = [1, 5, 20]

# The same number of gradient steps for every arm and every budget, so the
# comparison is about what survived rather than how much training happened.
RECOVERY_STEPS = 100
MEASURE_EVERY = 20

# Tasks the model has never seen. Order 01 uses 3, 0, 9, 5, 17, 1, 7 and 14,
# and task 15 is learned afterwards, so these are untouched. Each is learned
# from the same unlearned model with a fresh code and the same budget, which
# answers "cheap compared to what?". Several are used because tasks differ in
# difficulty and one would not be convincing.
UNSEEN_TASKS = ["2", "4", "6"]

# Hold the other seven tasks still during recovery, as the method would in
# normal use. Without this the generator could simply be overwritten and the
# comparison between arms would blur.
PROTECT_DURING_RECOVERY = True

SAVE_TO_R2 = True
RELEASE_GPU_WHEN_DONE = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Arms per budget: 3 | budgets:", BUDGETS, "| steps each:", RECOVERY_STEPS)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID`, and `R2_SECRET_ACCESS_KEY` through Colab Secrets
or environment variables.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Connected to R2 bucket:", BUCKET)

## 4. Load the study and the source model

In [ ]:
PREFIX = "uncle/paired_generalization/" + STUDY_ID
path = store.download(PREFIX + "/study.json", CACHE / "cache")
plan = json.loads(path.read_text())
if plan["code_version"] != CODE_VERSION:
    raise ValueError("This study requires a different fixed code revision.")
from uncle.learning_diagnostic import file_hash
if file_hash(REPO / "uncle/task_partition.json") != plan["partition_sha256"]:
    raise ValueError("Class partition differs from the saved study.")

storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
ALL_JOBS = jobs(plan)
ORDER_TASKS = tuple(plan["orders"][ORDER])
NEW_TASK = plan["new_tasks"][0]
CLASSES = plan["config"]["classes_per_task"]
EVAL_BATCH = plan["config"]["eval_batch_size"]

def fetch_tasks(include):
    """Build the task datasets, clearing a broken download if there is one."""
    import shutil, zipfile
    archive = Path(DATA_ROOT).parent / "tiny-imagenet-200.zip"
    for attempt in (1, 2):
        try:
            return build_tasks(root=DATA_ROOT,
                               partition_path=REPO / "uncle/task_partition.json",
                               partition_seed=42, include=include, download=True)
        except (zipfile.BadZipFile, EOFError, OSError) as problem:
            if attempt == 2:
                raise
            print("The Tiny ImageNet download is unusable:", problem)
            if archive.exists():
                head = archive.open("rb").read(4)
                print(f"   {archive} is {archive.stat().st_size} bytes, starts with {head!r}")
                archive.unlink()
            shutil.rmtree(Path(DATA_ROOT), ignore_errors=True)
            print("   Cleared it. Downloading again.")

tasks = fetch_tasks(ORDER_TASKS + tuple(UNSEEN_TASKS))

source_job = next(j for j in ALL_JOBS if j["id"] == f"sources/{ORDER}/seed_{SEED}")
branch_job = next(j for j in ALL_JOBS
                  if j["id"] == f"branches/{ORDER}/seed_{SEED}/forget_{FORGET_TASK}/learn_{NEW_TASK}")
state = storage.load(source_job)
if state is None or state["report"]["status"] != "complete":
    raise RuntimeError(f"Source model for seed {SEED} is not finished in R2.")

REQUESTS = tuple(("learn", t) for t in ORDER_TASKS)
REQUESTS += tuple(tuple(op) for op in branch_job["operations"])
PROTECT = [t for t in state["seen"] if t != FORGET_TASK]
print("Source:", source_job["id"], "| forgetting task", FORGET_TASK)

## 5. Measuring, and choosing the images

`inspect` reports accuracy and input sensitivity, which is how much the ten output scores
move between different images. `sample_indices` picks an equal number from each class.

In [ ]:
from torch.func import functional_call
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F

def inspect(hypernet, target, trainer, code_task, data_task):
    """Score the weights produced by `code_task` on `data_task`'s images.

    The two differ in the fresh-start arm, where a brand new code is trained
    on task 3's data, so its weights must still be judged on task 3's images.
    """
    # Evaluation mode matters more than usual here. In training mode
    # BatchNorm normalises using each batch's own statistics, which rescales a
    # nearly dead signal back up to unit size and hides exactly the collapse
    # this notebook is measuring. The previous mode is put back afterwards so
    # a measurement taken mid-training does not stop the training.
    was_training = hypernet.training
    hypernet.eval(); target.eval()
    weights = hypernet.weights_for(code_task)
    buffers = {n: v.clone() for n, v in trainer.task_buffers[code_task].items()}
    loader = DataLoader(tasks[data_task]["test"], batch_size=EVAL_BATCH)
    correct = total = 0
    lowest = highest = None
    with torch.no_grad():
        for images, labels in loader:
            scores = functional_call(target, (weights, buffers),
                                     (images.to(trainer.device),), strict=True)
            correct += (scores.argmax(1) == labels.to(trainer.device)).sum().item()
            total += labels.numel()
            low, high = scores.min(dim=0).values, scores.max(dim=0).values
            lowest = low if lowest is None else torch.minimum(lowest, low)
            highest = high if highest is None else torch.maximum(highest, high)
    if was_training:
        hypernet.train(); target.train()
    return {"accuracy": 100.0 * correct / total,
            "input_sensitivity": (highest - lowest).max().item()}

def sample_indices(dataset, per_class, seed):
    """An equal number of images from each class, chosen reproducibly.

    ClassTask keeps its labels in `.targets`, so this costs nothing: no image
    is opened just to find out which class it belongs to.
    """
    generator = torch.Generator().manual_seed(seed)
    chosen = []
    for label in range(CLASSES):
        here = [i for i, value in enumerate(dataset.targets) if value == label]
        order = torch.randperm(len(here), generator=generator)[:per_class]
        chosen.extend(here[int(i)] for i in order)
    return chosen

## 6. The recovery step

New code, not a copy: the study has no recovery operation. It is an ordinary supervised
fine-tune of the generator and one task code, with the method's own protection term so the
other seven tasks are held still.

In [ ]:
def recover(trainer, code_task, data_task, protected, indices, steps, on_step):
    """Train the generator and one task code on a small sample of images.

    This is new code, not a copy of anything in the repository: the study has
    no recovery operation. It is an ordinary supervised fine-tune, with the
    same protection term the method uses elsewhere so the other tasks are
    held still while this one is pulled back.
    """
    hypernet, config = trainer.hypernet, trainer.config
    snapshot = hypernet.snapshot()
    trainer._reference = {}

    code = hypernet.task_codes[code_task]
    trainable = [*hypernet.generator_parameters(), code]
    hypernet.requires_grad_(False)
    for parameter in trainable:
        parameter.requires_grad_(True)
    optimizer = torch.optim.Adam(trainable, lr=config.learning_rate)

    subset = Subset(tasks[data_task]["train"], indices)
    loader = DataLoader(subset, batch_size=min(config.batch_size, len(subset)),
                        shuffle=True, drop_last=False,
                        generator=torch.Generator().manual_seed(config.seed))

    hypernet.train(); trainer.target.train()
    on_step(0)
    step = 0
    while step < steps:
        for images, labels in loader:
            if step >= steps:
                break
            weights = hypernet.weights_for(code_task)
            scores = functional_call(trainer.target,
                                     (weights, trainer.task_buffers[code_task]),
                                     (images.to(trainer.device),), strict=True)
            loss = F.cross_entropy(scores, labels.to(trainer.device))
            if protected:
                loss = loss + config.beta * trainer.preserve(protected, snapshot)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            step += 1
            if step % MEASURE_EVERY == 0:
                with torch.no_grad():
                    on_step(step)
                hypernet.train(); trainer.target.train()
    trainer._reference = {}

## 7. Unlearn once, then try every arm

The model is unlearned a single time and that state is kept, so all nine runs start from
exactly the same place.

In [ ]:
import copy, gc
from uncle.study_training import run_request

def build():
    config = Config(**{**plan["config"], "seed": SEED, "device": DEVICE,
                       "requests": REQUESTS, "burn_in": UNLEARN_STEPS})
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=True)
    checkpoint.restore(state, hypernet=hypernet, uncle=trainer)
    restore_rng(state["all_rng"])
    return config, target, hypernet, trainer

# Unlearn once. Every arm below starts from this same model, so any difference
# between them comes from the arm and not from a different starting point.
with Dashboard("Unlearning once, then recovering") as board:
    board.note(f"unlearning task {FORGET_TASK} for {UNLEARN_STEPS} steps")
    config, target, hypernet, trainer = build()
    before = inspect(hypernet, target, trainer, FORGET_TASK, FORGET_TASK)
    run_request(trainer, "forget", FORGET_TASK, PROTECT, None, lambda row: None,
                lambda active: None, forget_interval=UNLEARN_STEPS,
                evaluation_interval=10 ** 9)
    after = inspect(hypernet, target, trainer, FORGET_TASK, FORGET_TASK)
    UNLEARNED = copy.deepcopy(hypernet.state_dict())
    BUFFERS = {t: {n: v.clone() for n, v in b.items()}
               for t, b in trainer.task_buffers.items()}
    FRESH_BUFFERS = {n: v.clone() for n, v in trainer.buffer_template.items()}
    print(f"\nbefore unlearning : {before['accuracy']:.1f}%  sensitivity {before['input_sensitivity']:.3e}")
    print(f"after unlearning  : {after['accuracy']:.1f}%  sensitivity {after['input_sensitivity']:.3e}")

    ARMS = [("attacker", True, False), ("fresh start", False, True), ("code only", True, True)]
    results = []
    work = board.bar(len(BUDGETS) * len(ARMS), "Recovery runs", unit="run")

    for budget in BUDGETS:
        indices = sample_indices(tasks[FORGET_TASK]["train"], budget, seed=1000 + budget)
        for arm, keep_code, reset_stats in ARMS:
            board.note(f"{arm}, {budget} image(s) per class")
            print(f"\n=== {arm}: {budget} image(s) per class, {len(indices)} images total ===")

            # Put the unlearned model back, so every arm starts identically.
            # A previous arm may have added a task code that the saved state
            # does not contain, and load_state_dict refuses an unexpected one,
            # so drop those first.
            for name in list(hypernet.task_codes):
                if f"task_codes.{name}" not in UNLEARNED:
                    del hypernet.task_codes[name]
            hypernet.load_state_dict(UNLEARNED)
            trainer.task_buffers = {t: {n: v.clone() for n, v in b.items()}
                                    for t, b in BUFFERS.items()}

            if keep_code:
                code_task = FORGET_TASK
            else:
                # A brand new code: this arm learns task 3 from nothing inside
                # the same generator, which is the bar the others must beat.
                code_task = f"{FORGET_TASK}_fresh_{budget}"
                if code_task in hypernet.task_codes:
                    del hypernet.task_codes[code_task]
                hypernet.add_task(code_task)
            if reset_stats:
                trainer.task_buffers[code_task] = {n: v.clone() for n, v in FRESH_BUFFERS.items()}
            elif code_task not in trainer.task_buffers:
                trainer.task_buffers[code_task] = {n: v.clone() for n, v in FRESH_BUFFERS.items()}

            rows = []
            def note(step, code_task=code_task):
                row = {"step": step, **inspect(hypernet, target, trainer, code_task, FORGET_TASK)}
                rows.append(row)
                print(f"    step {step:>3} | accuracy {row['accuracy']:5.1f}% | "
                      f"sensitivity {row['input_sensitivity']:.3e}")

            recover(trainer, code_task, FORGET_TASK,
                    PROTECT if PROTECT_DURING_RECOVERY else [],
                    indices, RECOVERY_STEPS, note)
            others = {t: inspect(hypernet, target, trainer, t, t)["accuracy"] for t in PROTECT}
            results.append({"arm": arm, "budget": budget, "images": len(indices),
                            "rows": rows, "other_tasks": others,
                            "others_mean": sum(others.values()) / len(others)})
            work.update(1)
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
    # The control: how fast does this same model learn a task it has never
    # seen, given the same budget? Nothing about task 3 is involved.
    unseen_work = board.bar(len(BUDGETS) * len(UNSEEN_TASKS), "Unseen tasks", unit="run")
    for budget in BUDGETS:
        for unseen in UNSEEN_TASKS:
            board.note(f"unseen task {unseen}, {budget} image(s) per class")
            print(f"\n=== unseen task {unseen}: {budget} image(s) per class ===")
            for name in list(hypernet.task_codes):
                if f"task_codes.{name}" not in UNLEARNED:
                    del hypernet.task_codes[name]
            hypernet.load_state_dict(UNLEARNED)
            trainer.task_buffers = {t: {n: v.clone() for n, v in b.items()}
                                    for t, b in BUFFERS.items()}

            code_task = f"unseen_{unseen}_{budget}"
            hypernet.add_task(code_task)
            trainer.task_buffers[code_task] = {n: v.clone() for n, v in FRESH_BUFFERS.items()}
            indices = sample_indices(tasks[unseen]["train"], budget, seed=2000 + budget)

            rows = []
            def note(step, code_task=code_task, unseen=unseen):
                row = {"step": step, **inspect(hypernet, target, trainer, code_task, unseen)}
                rows.append(row)
                print(f"    step {step:>4} | accuracy {row['accuracy']:5.1f}% | "
                      f"sensitivity {row['input_sensitivity']:.3e}")

            recover(trainer, code_task, unseen,
                    PROTECT if PROTECT_DURING_RECOVERY else [],
                    indices, RECOVERY_STEPS, note)
            results.append({"arm": f"unseen {unseen}", "budget": budget,
                            "images": len(indices), "rows": rows,
                            "other_tasks": {}, "others_mean": float("nan")})
            unseen_work.update(1)
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
    unseen_work.close()
    work.close()

print(f"\nFinished {len(results)} recovery runs.")

## 8. Compare

In [ ]:
by_key = {(r["arm"], r["budget"]): r for r in results}

def best(run):
    """The highest accuracy reached, not the last one.

    Over a long run on few images the model overfits and the final number
    drifts below the peak. For "could someone restore this task", the peak is
    the honest figure: it is what an attacker would stop at.
    """
    return max(r["accuracy"] for r in run["rows"])

ARM_NAMES = ("attacker", "fresh start", "code only")

print("BEST ACCURACY REACHED")
print(f"{'budget':>7} {'images':>7}  {'attacker':>10} {'fresh start':>12} {'code only':>10}   {'others kept':>11}")
print("-" * 68)
for budget in BUDGETS:
    row = [by_key[(arm, budget)] for arm in ARM_NAMES]
    peaks = [best(r) for r in row]
    print(f"{budget:>7} {row[0]['images']:>7}  {peaks[0]:9.1f}% {peaks[1]:11.1f}% {peaks[2]:9.1f}%"
          f"   {row[0]['others_mean']:10.1f}%")

print()
print("WHERE EACH ARM FINISHED (lower than the peak means it overfitted)")
print(f"{'budget':>7}  {'attacker':>10} {'fresh start':>12} {'code only':>10}")
print("-" * 45)
for budget in BUDGETS:
    row = [by_key[(arm, budget)] for arm in ARM_NAMES]
    finals = [r["rows"][-1]["accuracy"] for r in row]
    print(f"{budget:>7}  {finals[0]:9.1f}% {finals[1]:11.1f}% {finals[2]:9.1f}%")

print()
print(f"For reference: task {FORGET_TASK} scored {before['accuracy']:.1f}% before unlearning,")
print(f"and {after['accuracy']:.1f}% straight after. Chance is {100.0/CLASSES:.0f}%.")

print()
print("VERDICT")
chance = 100.0 / CLASSES
gaps, bn_gaps, any_learning = [], [], False
for budget in BUDGETS:
    a = best(by_key[("attacker", budget)])
    f = best(by_key[("fresh start", budget)])
    c = best(by_key[("code only", budget)])
    gaps.append(a - f)
    bn_gaps.append(a - c)
    any_learning = any_learning or max(a, f, c) > chance + 2.0

if not any_learning:
    print("  Nothing recovered in any arm. The learning signal has vanished.")
    print("  This does NOT show the information was removed, only that gradient")
    print("  descent could not reach it. Say exactly that, and no more.")
else:
    mean_gap = sum(gaps) / len(gaps)
    print(f"  Attacker beats a fresh start by {mean_gap:+.1f} points on average.")
    if mean_gap > 3.0:
        print("  Something about task 3 survived unlearning. The model recovers it")
        print("  faster than it learns it from nothing, using the same images and")
        print("  the same number of steps. A privacy claim for this method fails.")
    elif mean_gap < -3.0:
        print("  A fresh code does BETTER than the old one, so unlearning left the")
        print("  old code actively harder to train than an unused one.")
    else:
        print("  The two are close, so no usable trace of task 3 is left in the code.")
    mean_bn = sum(bn_gaps) / len(bn_gaps)
    print()
    print(f"  Keeping the BatchNorm statistics is worth {mean_bn:+.1f} points on average.")
    if mean_bn > 3.0:
        print("  Those statistics are derived from task 3's images and unlearning")
        print("  never touches them. That is a data remnant the method does not")
        print("  attempt to remove, and it is worth reporting on its own.")

print()
print("CHEAP COMPARED TO WHAT? tasks this model has never seen")
print(f"{'budget':>7}  {'task 3, fresh code':>19}  " +
      "  ".join(f"{'task ' + u:>8}" for u in UNSEEN_TASKS) + f"  {'their mean':>11}")
print("-" * (32 + 10 * len(UNSEEN_TASKS) + 13))
unseen_means = {}
for budget in BUDGETS:
    peaks = [best(by_key[(f"unseen {u}", budget)]) for u in UNSEEN_TASKS]
    unseen_means[budget] = sum(peaks) / len(peaks)
    own = best(by_key[("fresh start", budget)])
    print(f"{budget:>7}  {own:18.1f}%  " + "  ".join(f"{p:7.1f}%" for p in peaks)
          + f"  {unseen_means[budget]:10.1f}%")

print()
gap_unseen = sum(best(by_key[("fresh start", b)]) - unseen_means[b] for b in BUDGETS) / len(BUDGETS)
print(f"  Task 3 with a fresh code beats an unseen task by {gap_unseen:+.1f} points on average.")
if gap_unseen > 3.0:
    print("  Task 3 is easier for this model than a task it never saw, even from a")
    print("  brand new code. Knowledge of task 3 is still in the shared generator,")
    print("  which unlearning was supposed to remove. This is the privacy result.")
elif gap_unseen < -3.0:
    print("  An unseen task is EASIER than task 3, so task 3 is simply a hard task")
    print("  and no residual advantage is visible.")
else:
    print("  Task 3 is no easier than a task the model never saw. So the cheap")
    print("  recovery reflects how easily this model learns anything, not leftover")
    print("  knowledge of task 3. Say that plainly rather than claiming a leak.")
print(f"  Treat anything within 3 points as a tie: identical runs differ by about that much.")

print()
print("RECOVERY SPEED, attacker arm")
print(f"{'budget':>7} " + " ".join(f"{'step ' + str(s):>10}" for s in
      [r['step'] for r in by_key[('attacker', BUDGETS[0])]['rows']]))
print("-" * 68)
for budget in BUDGETS:
    row = by_key[("attacker", budget)]
    print(f"{budget:>7} " + " ".join(f"{r['accuracy']:9.1f}%" for r in row["rows"]))

## 9. Chart

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime, timezone

EXPORT = CACHE / "diagnostics" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EXPORT.mkdir(parents=True, exist_ok=True)

colours = {"attacker": "#b8322a", "fresh start": "#245d99", "code only": "#4c956c"}
figure, axes = plt.subplots(1, len(BUDGETS), figsize=(5 * len(BUDGETS), 4.4), sharey=True)
if len(BUDGETS) == 1:
    axes = [axes]

for axis, budget in zip(axes, BUDGETS):
    for arm in ("attacker", "fresh start", "code only"):
        run = by_key[(arm, budget)]
        axis.plot([r["step"] for r in run["rows"]], [r["accuracy"] for r in run["rows"]],
                  color=colours[arm], marker="o", linewidth=1.8, label=arm)
    axis.axhline(before["accuracy"], color="black", linestyle="--", linewidth=1,
                 label="before unlearning")
    axis.axhline(100.0 / CLASSES, color="grey", linestyle=":", linewidth=1, label="chance")
    axis.set_title(f"{budget} image(s) per class")
    axis.set_xlabel("recovery step")
axes[0].set_ylabel("task " + FORGET_TASK + " accuracy (%)")
axes[0].legend(fontsize=8)

figure.suptitle(f"{STUDY_ID} | seed {SEED} | recovering task {FORGET_TASK} after {UNLEARN_STEPS} unlearning steps")
figure.tight_layout()
figure.savefig(EXPORT / "recovery.png", dpi=180)
figure.savefig(EXPORT / "recovery.svg")
plt.show()
print("Figure written to", EXPORT)

## 10. Save

In [ ]:
import csv

summary = {
    "study_id": STUDY_ID, "code_version": CODE_VERSION, "order": ORDER, "seed": SEED,
    "forget_task": FORGET_TASK, "unlearn_steps": UNLEARN_STEPS,
    "budgets": BUDGETS, "recovery_steps": RECOVERY_STEPS,
    "unseen_tasks": UNSEEN_TASKS,
    "protected_during_recovery": PROTECT_DURING_RECOVERY,
    "before_unlearning": before, "after_unlearning": after,
    "classes_per_task": CLASSES, "beta": plan["config"]["beta"],
    "results": results,
    "reading": ("Every arm starts from the same unlearned model and gets the same images "
                "and the same number of steps. The attacker arm keeps task 3's code and "
                "BatchNorm statistics; fresh start keeps neither; code only keeps the "
                "code. Recovery images come from task 3's own training set."),
}
(EXPORT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

with (EXPORT / "recovery.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=["arm", "budget", "images", "step",
                                                "accuracy", "input_sensitivity"])
    writer.writeheader()
    for run in results:
        for row in run["rows"]:
            writer.writerow({"arm": run["arm"], "budget": run["budget"],
                             "images": run["images"], **row})

print("Saved locally:")
for item in sorted(EXPORT.iterdir()):
    print("  ", item)

if SAVE_TO_R2:
    destination = PREFIX + "/diagnostics/" + EXPORT.name
    with Dashboard("Saving results") as board:
        files = sorted(EXPORT.iterdir())
        sending = board.bar(len(files), "Upload diagnostics")
        for item in files:
            sending.set_postfix_str(item.name)
            store.upload(item, destination + "/" + item.name)
            sending.update(1)
        sending.close()
    print("Uploaded to", destination)
    SAVED_LOCATION = destination
else:
    SAVED_LOCATION = str(EXPORT)
    print("\nSAVE_TO_R2 is off, so the only copy is inside this runtime.")

EXPECTED_FILES = ["recovery.csv", "recovery.png", "recovery.svg", "summary.json"]
SAVE_VERIFIED = sorted(item.name for item in EXPORT.iterdir()) == EXPECTED_FILES
print("\nAll expected files present:", SAVE_VERIFIED)

## 11. Hand the GPU back

In [ ]:
if not SAVE_VERIFIED:
    print("Not releasing the GPU: some output files are missing.")
    print("Rerun the save section before closing this runtime.")
elif not RELEASE_GPU_WHEN_DONE or not IN_COLAB:
    print("Results saved to:", SAVED_LOCATION)
else:
    print("Results saved to:", SAVED_LOCATION)
    print("Share that path.")
    import gc
    for name in ("results", "state", "tasks", "UNLEARNED", "BUFFERS"):
        globals().pop(name, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    from google.colab import runtime
    print("Releasing the Colab runtime now.")
    runtime.unassign()

## What to tell your mentor

Send me the table and verdict from section 7. The headline is one sentence, and which
sentence it is depends on the gap between the attacker arm and the fresh-start arm.

**If the attacker arm wins**, you have a privacy result: the method's own success metric
says the task is forgotten, while a handful of its original images bring it back faster
than learning it from nothing would. The metrics in the paper cannot see this.

**If the BatchNorm gap is large**, you have a second, independent result: unlearning never
touches the running statistics, which are derived from the forgotten task's images, so a
data remnant is left behind by construction. This one does not even need the recovery to
succeed to be worth reporting.

**If nothing recovers**, say only that gradient descent could not reach the task from a
model that has stopped responding to its input. That is not the same as the information
being gone, and claiming otherwise would be the easiest thing for a reviewer to attack.

**What this does not cover.** One seed, one task, one order, recovery from the task's own
training images rather than unseen ones. Enough for a finding, not enough for a general
claim about the method.